# 00 · Funções e parâmetros compartilhados

Notebook de apoio carregado pelos demais com `%run ./00_funcoes.ipynb`. Reúne:

1. **Parâmetros padrão** (fuso, feriados, divisão temporal, janela, limiar, sementes);
2. **Preparação**: carga (UTC → Brasília), consolidação das chaves repetidas e agregação por recurso × janela;
3. **Perfil de normalidade contextual** (mediana/MAD por recurso × tipo de dia × hora) e decisão **Normal/Residual**;
4. **Baselines** (z-score global, percentil global, semana anterior);
5. **Regimes** (k-means descritivo);
6. **Injeção de falhas**;
7. **Avaliação** e **consulta pré-incidente**;
8. **Estilo dos gráficos**.

Toda a análise é **por recurso** (`CD_RCS_CAT`) e há apenas dois estados: **Normal** (qualquer regime normal esperado no contexto) e **Residual** (todo o resto).

In [ ]:
import json
import warnings
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.metrics import (adjusted_rand_score, average_precision_score, calinski_harabasz_score,
                             davies_bouldin_score, precision_recall_fscore_support, silhouette_score)
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

## 1. Parâmetros padrão
Cada notebook pode sobrescrever estes valores na sua célula de parâmetros.

In [ ]:
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ARQUIVO_DADOS = RAIZ.parent / "DADOS" / "APIs-Agr-20260801-20260908_hash.csv"  # dados na raiz do projeto
RESULTADOS = RAIZ / "resultados"
FIGURAS = RESULTADOS / "figuras"
FIGURAS.mkdir(parents=True, exist_ok=True)

FUSO = "America/Sao_Paulo"          # TS_EVT vem em UTC
FERIADOS = ["2026-09-07"]           # feriados usam o perfil de domingo
JANELA = "1h"
JANELAS_SENSIBILIDADE = ["15min", "30min", "1h", "4h"]

# Divisão temporal no horário de Brasília (fim exclusivo). O τ é calibrado por validação
# cruzada semanal dentro do TREINO (ver ajustar_metodos).
TREINO = (pd.Timestamp("2026-08-01"), pd.Timestamp("2026-09-01"))
TESTE = (pd.Timestamp("2026-09-01"), pd.Timestamp("2026-09-08"))

META_ALARMES = 0.01                 # fração das janelas de agosto (fora da amostra) marcadas como Residual
Q_ATRIBUTO = 0.99                   # quantil que define o limite de cada atributo no recurso
TAU_MIN = 3.0                       # limite mínimo de cada atributo (escala z)
TAU_NORM_MIN = 1.0                  # τ mínimo sobre o escore normalizado
MIN_AMOSTRAS_SLOT = 5               # abaixo disso o slot usa a reserva recurso × hora
HORAS_ANTES = 6                     # período analisado antes do registro do incidente
SEMENTES = list(range(10))
K_MAX = 8

## 2. Preparação dos dados

In [ ]:
COLUNAS = {"TS_EVT": "ts", "CD_RCS_CAT": "recurso", "CD_RPST": "status",
           "QT_TTL_TRAN": "qtd", "QT_TMP_MEDI_EXEA": "tmp", "CD_CLI": "cliente"}

# Atributos de comportamento usados na decisão Normal/Residual
ATRIBUTOS = ["log_vol", "tx_4xx", "tx_5xx", "log_lat_2xx", "log_n_cli", "top1_vol", "frac_cli_4xx", "frac_cli_5xx"]
NOMES = {"log_vol": "volume", "tx_4xx": "taxa 4xx", "tx_5xx": "taxa 5xx", "log_lat_2xx": "latência 2xx",
         "log_n_cli": "nº de clientes", "top1_vol": "part. maior cliente",
         "frac_cli_4xx": "frac. clientes c/ 4xx", "frac_cli_5xx": "frac. clientes c/ 5xx"}
NOMES_DIA = {"util": "dia útil", "sabado": "sábado", "domingo_feriado": "domingo/feriado", "nao_util": "não útil"}
TIPOS_ORDEM = ["util", "sabado", "domingo_feriado", "nao_util"]


def carregar(caminho=ARQUIVO_DADOS, fuso=FUSO):
    """Lê o CSV e converte TS_EVT de UTC para o horário local do negócio."""
    df = pd.read_csv(caminho, usecols=list(COLUNAS), parse_dates=["TS_EVT"]).rename(columns=COLUNAS)
    df["ts"] = df["ts"].dt.tz_localize("UTC").dt.tz_convert(fuso).dt.tz_localize(None)
    return df


def consolidar(df):
    """Junta linhas com a mesma chave (minuto, recurso, status, cliente): soma o volume e
    usa a média de latência ponderada pelo volume."""
    df = df.assign(tmp_total=df["qtd"] * df["tmp"])
    out = df.groupby(["ts", "recurso", "status", "cliente"], as_index=False, sort=False).agg(
        qtd=("qtd", "sum"), tmp_total=("tmp_total", "sum"))
    out["tmp"] = out["tmp_total"] / out["qtd"]
    return out.drop(columns="tmp_total")


def tipo_dia(ts, feriados, categorias=3):
    """útil / sábado / domingo_feriado (ou útil / nao_util com categorias=2)."""
    feriado = ts.dt.normalize().isin(pd.to_datetime(list(feriados)))
    dow = ts.dt.dayofweek
    if categorias == 2:
        return np.where((dow >= 5) | feriado, "nao_util", "util")
    return np.select([feriado | (dow == 6), dow == 5], ["domingo_feriado", "sabado"], "util")


def limites_recursos(df, janela):
    """Primeira janela e fim (exclusivo) da última janela completa de cada recurso."""
    g = df.groupby("recurso")["ts"].agg(["min", "max"])
    return pd.DataFrame({"inicio": g["min"].dt.floor(janela),
                         "fim": (g["max"] + pd.Timedelta("1min")).dt.floor(janela)})


def agregar(df, janela, feriados=FERIADOS, limites=None):
    """Agrega por recurso × janela. Janelas sem registro dentro de ``limites`` entram com volume zero."""
    if limites is None:
        limites = limites_recursos(df, janela)
    df = df.assign(janela=df["ts"].dt.floor(janela), familia=df["status"] // 100, tmp_total=df["qtd"] * df["tmp"])
    for fam in (2, 4, 5):
        df[f"q{fam}"] = df["qtd"].where(df["familia"] == fam, 0)
    df["t2"] = df["tmp_total"].where(df["familia"] == 2, 0)

    niveis = ["recurso", "janela"]
    cli = df.groupby(["recurso", "janela", "cliente"], sort=False)[["qtd", "q4", "q5"]].sum()
    cli["tem4"], cli["tem5"] = cli["q4"] > 0, cli["q5"] > 0
    cli_ag = cli.groupby(level=niveis).agg(n_cli=("qtd", "size"), max_cli=("qtd", "max"),
                                           n_cli_4xx=("tem4", "sum"), n_cli_5xx=("tem5", "sum"))
    # Participação dos 2 clientes com mais erros no total de erros de cada família
    for fam in ("q4", "q5"):
        top2 = cli[fam].sort_values(ascending=False).groupby(level=niveis).head(2)
        cli_ag[f"top2_{fam}"] = top2.groupby(level=niveis).sum()

    jan = df.groupby(niveis)[["qtd", "q2", "q4", "q5", "t2", "tmp_total"]].sum()
    jan = jan.rename(columns={"qtd": "vol", "tmp_total": "t_total"}).join(cli_ag)

    grades = [pd.MultiIndex.from_product([[r], pd.date_range(l.inicio, l.fim - pd.Timedelta(janela), freq=janela)],
                                         names=niveis) for r, l in limites.iterrows()]
    jan = jan.reindex(grades[0].append(grades[1:])).fillna(0).reset_index()

    vol, n_cli = jan["vol"].replace(0, np.nan), jan["n_cli"].replace(0, np.nan)
    jan["tx_4xx"] = (jan["q4"] / vol).fillna(0)
    jan["tx_5xx"] = (jan["q5"] / vol).fillna(0)
    jan["lat_2xx"] = jan["t2"] / jan["q2"].replace(0, np.nan)
    jan["top1_vol"] = (jan["max_cli"] / vol).fillna(0)
    jan["frac_cli_4xx"] = (jan["n_cli_4xx"] / n_cli).fillna(0)
    jan["frac_cli_5xx"] = (jan["n_cli_5xx"] / n_cli).fillna(0)
    jan["conc_4xx"] = (jan["top2_q4"] / jan["q4"].replace(0, np.nan)).fillna(0)
    jan["conc_5xx"] = (jan["top2_q5"] / jan["q5"].replace(0, np.nan)).fillna(0)
    jan["log_vol"] = np.log1p(jan["vol"])
    jan["log_n_cli"] = np.log1p(jan["n_cli"])
    jan["log_lat_2xx"] = np.log1p(jan["lat_2xx"])

    jan["hora"] = jan["janela"].dt.hour
    jan["tipo_dia"] = tipo_dia(jan["janela"], feriados, 3)       # proposto
    jan["tipo_dia2"] = tipo_dia(jan["janela"], feriados, 2)      # comparação: útil / não útil
    jan["tipo_dia_sf"] = tipo_dia(jan["janela"], [], 3)          # comparação: feriado tratado como dia útil
    jan["tipo_dia2_sf"] = tipo_dia(jan["janela"], [], 2)
    return adicionar_tipo_dia_adaptado(jan.sort_values(niveis).reset_index(drop=True))


def recursos_3_tipos():
    """Recursos em que sábado e domingo diferem (decidido no notebook 03 só com dados de agosto)."""
    arq = RESULTADOS / "03_resumo.json"
    return set(json.loads(arq.read_text())["recursos_3_tipos"]) if arq.exists() else None


def adicionar_tipo_dia_adaptado(jan):
    """Tipo de dia do método proposto: 3 categorias onde sábado ≠ domingo; senão útil / não útil.
    O feriado sempre acompanha o domingo (ou o 'não útil')."""
    r3 = recursos_3_tipos()
    if r3 is None:
        jan["tipo_dia_ad"], jan["tipo_dia_ad_sf"] = jan["tipo_dia"], jan["tipo_dia_sf"]
        return jan
    tres = jan["recurso"].isin(r3)
    jan["tipo_dia_ad"] = np.where(tres, jan["tipo_dia"], jan["tipo_dia2"])
    jan["tipo_dia_ad_sf"] = np.where(tres, jan["tipo_dia_sf"], jan["tipo_dia2_sf"])
    return jan


def recortar(jan, periodo, col="janela"):
    return jan[(jan[col] >= periodo[0]) & (jan[col] < periodo[1])].copy()

## 3. Perfil de normalidade contextual e decisão Normal/Residual

Para cada recurso × tipo de dia × hora guardam-se a **mediana** e a **escala robusta** (1,4826 × MAD) de cada atributo. Cada janela recebe um **z robusto** por atributo, contado só na **direção desfavorável** (volume e nº de clientes nos dois sentidos; os demais só na alta). Slots com menos de 5 amostras usam a reserva recurso × hora. Um piso na escala evita z infinito em atributos que quase não variam.

**Limite por atributo.** Os atributos têm caudas muito diferentes: em agosto, as taxas de 4xx/5xx de alguns recursos têm picos reais frequentes (z > 50), enquanto volume e latência raramente passam de z = 10. Por isso, o z de cada atributo é dividido pelo **limite daquele atributo no recurso** (quantil 99% dos seus z fora da amostra em agosto, mínimo 3). O escore da janela é o maior desses valores normalizados, e a janela é **Residual** se ele passa do τ do recurso.

In [ ]:
DIRECAO = {a: "alta" for a in ATRIBUTOS}
DIRECAO.update(log_vol="ambos", log_n_cli="ambos")
ESCALA_MIN = pd.Series({"log_vol": 0.10, "tx_4xx": 0.01, "tx_5xx": 0.01, "log_lat_2xx": 0.05,
                        "log_n_cli": 0.10, "top1_vol": 0.03, "frac_cli_4xx": 0.03, "frac_cli_5xx": 0.03})


def z_direcional(z):
    out = pd.DataFrame(index=z.index)
    for a in ATRIBUTOS:
        out[a] = z[a].abs() if DIRECAO[a] == "ambos" else z[a].clip(lower=0)
    return out.fillna(0)


def _escala(dados, chaves, mediana):
    x = dados.set_index(chaves)[ATRIBUTOS]
    mad = (x - mediana.reindex(x.index).to_numpy()).abs().groupby(level=chaves).median()
    return (1.4826 * mad).clip(lower=ESCALA_MIN, axis=1)


class PerfilContextual:
    """Mediana/MAD por recurso × contexto. ``contexto=()`` equivale ao z-score global por recurso."""

    minimo = TAU_MIN

    def __init__(self, contexto=("tipo_dia", "hora"), min_amostras=MIN_AMOSTRAS_SLOT):
        self.chaves = ["recurso", *contexto]
        self.min_amostras = min_amostras

    def ajustar(self, treino, historico=None):
        g = treino.groupby(self.chaves)[ATRIBUTOS]
        self.mediana = g.median()
        self.escala = _escala(treino, self.chaves, self.mediana)
        self.q05, self.q95 = g.quantile(0.05), g.quantile(0.95)
        self.n = g.size()
        self.reserva = None
        if len(self.chaves) > 2:  # contexto com tipo de dia: reserva por recurso × hora
            ch = ["recurso", "hora"]
            g2 = treino.groupby(ch)[ATRIBUTOS]
            med2 = g2.median()
            self.reserva = (med2, _escala(treino, ch, med2), g2.quantile(0.05), g2.quantile(0.95))
        return self

    def referencia(self, jan):
        """Mediana, escala, p5 e p95 esperados para cada janela."""
        idx = pd.MultiIndex.from_frame(jan[self.chaves]) if len(self.chaves) > 1 else pd.Index(jan["recurso"])
        refs = [t.reindex(idx) for t in (self.mediana, self.escala, self.q05, self.q95)]
        if self.reserva is not None:
            poucas = self.n.reindex(idx).fillna(0).to_numpy() < self.min_amostras
            if poucas.any():
                idx2 = pd.MultiIndex.from_frame(jan.loc[poucas, ["recurso", "hora"]])
                pos = np.flatnonzero(poucas)
                for r, src in zip(refs, self.reserva):
                    r.iloc[pos] = src.reindex(idx2).to_numpy()
        return [r.set_axis(jan.index) for r in refs]

    def z(self, jan):
        med, esc, _, _ = self.referencia(jan)
        return (jan[ATRIBUTOS] - med) / esc

    def atributos(self, jan, historico=None):
        """Escore por atributo (z robusto na direção desfavorável)."""
        return z_direcional(self.z(jan))


def calibrar_tau(escores, recursos, meta=META_ALARMES, minimo=TAU_NORM_MIN):
    """τ por recurso: quantil (1 − meta) dos escores fora da amostra, com piso ``minimo``."""
    q = escores.groupby(recursos.to_numpy()).quantile(1 - meta)
    return q.clip(lower=minimo)


def sequencias(residual, recurso):
    """Id da sequência de janelas residuais consecutivas do mesmo recurso (−1 se Normal).
    Pressupõe ordenação por recurso e janela, sem lacunas na grade."""
    r, rec = np.asarray(residual, bool), np.asarray(recurso)
    nova = r & (np.r_[True, ~r[:-1]] | np.r_[True, rec[1:] != rec[:-1]])
    return np.where(r, np.cumsum(nova) - 1, -1)

## 4. Baselines

- **z-score global**: mesmo cálculo robusto, mas sem contexto (um perfil por recurso).
- **Percentil global p1–p99**: a janela sai da faixa [p1, p99] do treino de algum atributo, sem contexto. Escore = excesso relativo + 1 (1 = na borda da faixa).
- **Semana anterior**: z robusto da diferença para a mesma janela 7 dias antes.

Todos os métodos usam a **mesma normalização por atributo** (limite de cada atributo no recurso) e o limiar calibrado da mesma forma, de modo que diferem só na referência de normalidade: **τ por recurso** = quantil (1 − meta) dos escores **fora da amostra** em agosto, obtidos por validação cruzada com 4 dobras semanais (ajusta em 3 semanas, pontua a restante). O modelo final é ajustado com agosto inteiro. O teste (setembro) nunca participa.

In [ ]:
class PercentilGlobal:
    minimo = 1.0

    def ajustar(self, treino, historico=None):
        g = treino.groupby("recurso")[ATRIBUTOS]
        self.p01, self.p50, self.p99 = g.quantile(0.01), g.median(), g.quantile(0.99)
        return self

    def atributos(self, jan, historico=None):
        p01, p50, p99 = (t.reindex(jan["recurso"]).set_axis(jan.index) for t in (self.p01, self.p50, self.p99))
        x = jan[ATRIBUTOS]
        acima = (x - p99) / (p99 - p50).clip(lower=ESCALA_MIN, axis=1)
        abaixo = (p01 - x) / (p50 - p01).clip(lower=ESCALA_MIN, axis=1)
        esc = pd.DataFrame({a: np.maximum(acima[a], abaixo[a]) if DIRECAO[a] == "ambos" else acima[a]
                            for a in ATRIBUTOS}, index=jan.index)
        return (esc.fillna(-1) + 1).clip(lower=0)


class SemanaAnterior:
    minimo = TAU_MIN

    @staticmethod
    def _dif(jan, historico):
        h = historico.set_index(["recurso", "janela"])[ATRIBUTOS]
        ant = pd.MultiIndex.from_arrays([jan["recurso"], jan["janela"] - pd.Timedelta("7D")])
        return jan[ATRIBUTOS] - h.reindex(ant).set_axis(jan.index)

    def ajustar(self, treino, historico):
        d = self._dif(treino, historico).assign(recurso=treino["recurso"].to_numpy()).dropna(subset=["log_vol"])
        mad = d.groupby("recurso")[ATRIBUTOS].agg(lambda s: (s - s.median()).abs().median())
        self.escala = (1.4826 * mad).clip(lower=ESCALA_MIN, axis=1)
        return self

    def atributos(self, jan, historico):
        esc = self.escala.reindex(jan["recurso"]).set_axis(jan.index)
        dif = self._dif(jan, historico)
        # sem a semana anterior (1ª semana de agosto) o escore fica indefinido
        return z_direcional(dif / esc).where(dif["log_vol"].notna())


METODOS = {
    "Perfil contextual (proposto)": lambda: PerfilContextual(("tipo_dia_ad", "hora")),
    "Perfil 3 tipos de dia": lambda: PerfilContextual(("tipo_dia", "hora")),
    "Perfil 2 tipos de dia": lambda: PerfilContextual(("tipo_dia2", "hora")),
    "Perfil só hora": lambda: PerfilContextual(("hora",)),
    "Perfil sem feriado": lambda: PerfilContextual(("tipo_dia_ad_sf", "hora")),
    "z-score global": lambda: PerfilContextual(()),
    "Percentil global p1–p99": lambda: PercentilGlobal(),
    "Semana anterior": lambda: SemanaAnterior(),
}
PROPOSTO = "Perfil contextual (proposto)"
BASELINES = ["z-score global", "Percentil global p1–p99", "Semana anterior"]


def dobras_semanais(tr):
    """Dobra de validação cruzada: semanas de agosto 1–7, 8–14, 15–21 e 22–31."""
    return ((tr["janela"] - TREINO[0]).dt.days // 7).clip(upper=3).to_numpy()


def ajustar_metodos(jan, metodos=METODOS, meta=META_ALARMES, q_atributo=Q_ATRIBUTO):
    """Para cada método, com escores por atributo fora da amostra em agosto (4 dobras semanais):
    1. limite de cada atributo no recurso = quantil ``q_atributo`` (mínimo ``modelo.minimo``);
    2. escore normalizado = maior (escore do atributo / limite do atributo);
    3. τ por recurso = quantil (1 − meta) do escore normalizado (mínimo 1).
    O modelo final é ajustado com agosto inteiro."""
    tr = recortar(jan, TREINO)
    dobra = dobras_semanais(tr)
    rec = tr["recurso"].to_numpy()
    saida = {}
    for nome, fab in metodos.items():
        att = pd.DataFrame(np.nan, index=tr.index, columns=ATRIBUTOS)
        for k in range(4):
            m = fab().ajustar(tr[dobra != k], jan)
            att.loc[dobra == k] = m.atributos(tr[dobra == k], jan).to_numpy()
        final = fab().ajustar(tr, jan)
        limite = att.groupby(rec).quantile(q_atributo).clip(lower=final.minimo)
        esc = (att / limite.reindex(rec).to_numpy()).max(axis=1, skipna=False)
        saida[nome] = dict(modelo=final, limite=limite, tau=calibrar_tau(esc, tr["recurso"], meta),
                           escores_cal=esc, recursos_cal=tr["recurso"])
    return saida


def normalizado(a, jan, historico=None):
    """Escore de cada atributo dividido pelo seu limite no recurso."""
    att = a["modelo"].atributos(jan, historico)
    return att / a["limite"].reindex(jan["recurso"]).to_numpy()


def pontuar_metodo(a, jan, historico=None):
    """Escore da janela (maior atributo normalizado)."""
    return normalizado(a, jan, historico).max(axis=1, skipna=False)

## 5. Lado provável (matriz heurística)

Aplicada à janela de maior escore de uma sequência residual. Um atributo é **evidência** quando passa do seu limite no recurso (escore normalizado > 1). Os atributos são examinados em ordem decrescente de escore normalizado, e vale a primeira regra aplicável. **Espalhado** = a fração de clientes com erro também passa do limite e há pelo menos 3 clientes com erro; senão, **concentrado** se os 2 maiores clientes têm ≥ 80% dos erros.

| Atributo acima do limite | Abrangência | Lado provável |
|---|---|---|
| Latência 2xx | qualquer | provedor |
| Taxa ou fração de clientes com 5xx (4xx) | espalhado | provedor |
| Taxa ou fração de clientes com 5xx (4xx) | concentrado | consumidor |
| Volume em queda | — | provedor |
| Volume em alta e maior cliente concentrando | concentrado | consumidor |
| Nenhuma regra aplicável, ou recurso com < 3 clientes (regras de abrangência) | — | indeterminado |

In [ ]:
def lado_provavel(zn, zs, linha, poucos_clientes, lim=1.0):
    """zn: escore normalizado por atributo; zs: z com sinal (só para o sentido do volume)."""
    for a in zn.sort_values(ascending=False).index:
        if not zn[a] > lim:
            break
        if a == "log_lat_2xx":
            return "provedor", "latência 2xx elevada"
        if a in ("tx_5xx", "frac_cli_5xx", "tx_4xx", "frac_cli_4xx"):
            fam = "5xx" if "5xx" in a else "4xx"
            if poucos_clientes:
                return "indeterminado", f"{fam} elevado em recurso com poucos clientes"
            if zn[f"frac_cli_{fam}"] > lim and linha[f"n_cli_{fam}"] >= 3:
                return "provedor", f"{fam} elevado e espalhado entre clientes"
            if linha[f"conc_{fam}"] >= 0.8:
                return "consumidor", f"{fam} elevado e concentrado em até 2 clientes"
            continue
        if a in ("log_vol", "top1_vol", "log_n_cli"):
            if a != "top1_vol" and zs["log_vol"] < 0 and zn["log_vol"] > lim:
                return "provedor", "queda de volume"
            if zs["log_vol"] > 0 and zn["top1_vol"] > lim and not poucos_clientes:
                return "consumidor", "aumento de volume concentrado no maior cliente"
    return "indeterminado", "nenhuma regra aplicável"


def recursos_poucos_clientes(treino, minimo=3):
    return set(treino.groupby("recurso")["n_cli"].median().loc[lambda s: s < minimo].index)

## 6. Regimes (k-means descritivo)

O k-means usa **só atributos de comportamento** (padronizados). O número de regimes (k de 2 a `K_MAX`) é escolhido pelo silhouette. Depois, cada regime é nomeado a partir do cruzamento com o contexto (tipo de dia e horas em que predomina) e do seu nível de volume.

In [ ]:
def matriz_regimes(jan):
    x = jan[ATRIBUTOS].copy()
    x["log_lat_2xx"] = x["log_lat_2xx"].fillna(x["log_lat_2xx"].median())
    return x.to_numpy()


def kmeans_por_silhouette(x, k_max=K_MAX, semente=0):
    linhas, melhor = [], None
    for k in range(2, k_max + 1):
        km = KMeans(n_clusters=k, n_init=10, random_state=semente).fit(x)
        s = silhouette_score(x, km.labels_)
        linhas.append(dict(k=k, silhouette=s, inercia=km.inertia_))
        if melhor is None or s > melhor[0]:
            melhor = (s, km)
    return melhor[1], pd.DataFrame(linhas)


def faixas_horas(horas):
    """[0,1,2,9,10] -> '0–2h, 9–10h'."""
    horas = sorted(horas)
    if not horas:
        return "—"
    partes, ini, ant = [], horas[0], horas[0]
    for h in horas[1:] + [None]:
        if h is not None and h == ant + 1:
            ant = h
            continue
        partes.append(f"{ini}h" if ini == ant else f"{ini}–{ant}h")
        if h is not None:
            ini = ant = h
    return ", ".join(partes)


def nomear_regimes(g, rot):
    """Nome de cada regime a partir do volume relativo, erros e do contexto em que predomina."""
    g = g.assign(regime=rot)
    med_vol = g.groupby("regime")["vol"].median()
    ordem = med_vol.rank(method="first")
    n = len(med_vol)
    nomes = {}
    for c, gg in g.groupby("regime"):
        nivel = "alto" if ordem[c] > 2 * n / 3 else ("baixo" if ordem[c] <= n / 3 else "médio")
        dias = gg["tipo_dia"].value_counts(normalize=True)
        dia = NOMES_DIA[dias.index[0]] if dias.iloc[0] >= 0.7 else "vários dias"
        frac_hora = g.groupby("hora")["regime"].apply(lambda s: (s == c).mean())
        horas = faixas_horas(list(frac_hora[frac_hora >= 0.5].index))
        extra = ""
        if gg["tx_5xx"].median() > max(0.02, 3 * g["tx_5xx"].median()):
            extra = ", 5xx alto"
        elif gg["tx_4xx"].median() > max(0.05, 2 * g["tx_4xx"].median()) :
            extra = ", 4xx alto"
        nomes[c] = f"R{c}: volume {nivel}{extra} · {dia}" + (f" · {horas}" if horas != "—" else " · esporádico")
    return nomes

## 7. Injeção de falhas

Aplicada nos registros por minuto × cliente do período de teste, antes da agregação, para que os atributos por cliente reajam como numa falha real.

In [ ]:
TIPOS = {  # tipo -> (lado, {intensidade: fator})
    "latencia_provedor": ("provedor", {"baixa": 1.5, "media": 2.5, "alta": 4.0}),
    "erro_5xx_provedor": ("provedor", {"baixa": 0.05, "media": 0.15, "alta": 0.40}),
    "queda_volume": ("provedor", {"baixa": 0.70, "media": 0.50, "alta": 0.20}),
    "erro_4xx_consumidor": ("consumidor", {"baixa": 0.30, "media": 0.60, "alta": 0.90}),
    "rajada_consumidor": ("consumidor", {"baixa": 2.0, "media": 4.0, "alta": 8.0}),
}
INTENSIDADES = ["baixa", "media", "alta"]
NOMES_TIPO = {"latencia_provedor": "Latência elevada", "erro_5xx_provedor": "Pico de 5xx",
              "queda_volume": "Queda de volume", "erro_4xx_consumidor": "4xx de 1 cliente",
              "rajada_consumidor": "Rajada de 1 cliente"}


def limites_teste(limites, janela="1h"):
    """Período de teste de cada recurso, limitado ao fim dos seus dados (horas inteiras)."""
    fim = limites["fim"].clip(upper=TESTE[1]).dt.floor("1h")
    return pd.DataFrame({"inicio": TESTE[0], "fim": fim}, index=limites.index)


def planejar(lim_teste, semente, duracoes_h=(1, 2, 3, 4, 6), bloco_h=30):
    """Um episódio por bloco de ``bloco_h`` horas do teste de cada recurso, sem sobreposição.
    As 15 combinações tipo × intensidade são distribuídas em rodízio embaralhado."""
    rng = np.random.default_rng(semente)
    combos = [(t, i) for t in TIPOS for i in INTENSIDADES]
    fila, eps = [], []
    for recurso, l in lim_teste.iterrows():
        t = l["inicio"]
        while t + pd.Timedelta(hours=bloco_h) <= l["fim"]:
            if not fila:
                fila = [combos[k] for k in rng.permutation(len(combos))]
            tipo, intens = fila.pop()
            dur = int(rng.choice(duracoes_h))
            ini = t + pd.Timedelta(hours=2 + int(rng.integers(0, bloco_h - dur - 4 + 1)))
            eps.append(dict(recurso=recurso, tipo=tipo, intensidade=intens, lado=TIPOS[tipo][0],
                            inicio=ini, fim=ini + pd.Timedelta(hours=dur), duracao_h=dur))
            t += pd.Timedelta(hours=bloco_h)
    out = pd.DataFrame(eps)
    out.insert(0, "episodio", range(len(out)))
    return out


def _sorteia_cliente(sub, rng):
    vol = sub.groupby("cliente")["qtd"].sum()
    return rng.choice(vol.index, p=(vol / vol.sum()).to_numpy())


def aplicar_falhas(brutos, episodios, semente):
    rng = np.random.default_rng(1000 + semente)
    df = brutos.copy()
    novas = []
    for ep in episodios.itertuples():
        m = ((df["recurso"] == ep.recurso) & (df["ts"] >= ep.inicio) & (df["ts"] < ep.fim)).to_numpy()
        if not m.any():
            continue
        f = TIPOS[ep.tipo][1][ep.intensidade]
        sucesso = m & (df["status"] // 100 == 2).to_numpy()
        if ep.tipo == "latencia_provedor":
            df.loc[m, "tmp"] *= f
        elif ep.tipo == "erro_5xx_provedor":
            mov = rng.binomial(df.loc[sucesso, "qtd"].to_numpy(), f)
            df.loc[sucesso, "qtd"] -= mov
            nov = df.loc[sucesso].assign(qtd=mov, status=503)
            novas.append(nov[nov["qtd"] > 0])
        elif ep.tipo == "erro_4xx_consumidor":
            alvo = sucesso & (df["cliente"] == _sorteia_cliente(df.loc[m], rng)).to_numpy()
            mov = rng.binomial(df.loc[alvo, "qtd"].to_numpy(), f)
            df.loc[alvo, "qtd"] -= mov
            nov = df.loc[alvo].assign(qtd=mov, status=400)
            nov["tmp"] *= 0.5  # rejeição na validação costuma ser mais rápida
            novas.append(nov[nov["qtd"] > 0])
        elif ep.tipo == "queda_volume":
            df.loc[m, "qtd"] = rng.binomial(df.loc[m, "qtd"].to_numpy(), f)
        elif ep.tipo == "rajada_consumidor":
            alvo = m & (df["cliente"] == _sorteia_cliente(df.loc[m], rng)).to_numpy()
            df.loc[alvo, "qtd"] = np.round(df.loc[alvo, "qtd"] * f).astype(int)
    df = pd.concat([df, *novas], ignore_index=True)
    return df[df["qtd"] > 0].reset_index(drop=True)


def rotular(jan, episodios, janela):
    """Episódio que cobre ≥ 50% da janela (−1 = limpa; −2 = cobertura parcial, fora das métricas)."""
    passo = pd.Timedelta(janela).to_timedelta64()
    ep_id = np.full(len(jan), -1)
    ini = jan["janela"].to_numpy()
    rec = jan["recurso"].to_numpy()
    for ep in episodios.itertuples():
        m = rec == ep.recurso
        sobre = np.minimum(ini + passo, np.datetime64(ep.fim)) - np.maximum(ini, np.datetime64(ep.inicio))
        frac = np.clip(sobre / passo, 0, None)
        ep_id[m & (frac >= 0.5)] = ep.episodio
        ep_id[m & (frac > 0) & (frac < 0.5) & (ep_id == -1)] = -2
    return jan.assign(episodio=ep_id)


def montar_teste(brutos_teste, jan_limpo, episodios, janela, lim_teste, semente):
    """Série completa da semente: janelas limpas fora do teste + teste reagregado com falhas."""
    inj = aplicar_falhas(brutos_teste, episodios, semente)
    lim = lim_teste.copy()
    lim["fim"] = lim["fim"].dt.floor(janela)
    jt = agregar(inj, janela, FERIADOS, lim)
    fora = jan_limpo[~jan_limpo.set_index(["recurso", "janela"]).index.isin(jt.set_index(["recurso", "janela"]).index)]
    js = pd.concat([fora, jt], ignore_index=True).sort_values(["recurso", "janela"]).reset_index(drop=True)
    return rotular(js, episodios, janela)

## 8. Avaliação e consulta pré-incidente

In [ ]:
def avaliar_deteccao(jt, escore, tau, episodios):
    """Métricas por janela e por episódio no período de teste (``jt`` ordenado por recurso e janela)."""
    razao = escore / tau.reindex(jt["recurso"]).to_numpy()
    residual = (razao > 1).to_numpy()
    lab = jt["episodio"].to_numpy()
    ok = lab != -2
    y = lab[ok] >= 0
    p, r, f, _ = precision_recall_fscore_support(y, residual[ok], average="binary", zero_division=0)
    seq = sequencias(residual, jt["recurso"])
    s = pd.DataFrame({"seq": seq, "lab": lab})[seq >= 0]
    por_seq = s.groupby("seq")["lab"].agg(lambda v: (v >= 0).any() or (v == -2).any())
    fp_seq = int((~por_seq).sum())
    com_janela = set(np.unique(lab[lab >= 0]))
    detectados = set(np.unique(lab[(lab >= 0) & residual]))
    rec_ep = len(detectados) / max(len(com_janela), 1)
    prec_ep = len(detectados) / max(len(detectados) + fp_seq, 1)
    f1_ep = 2 * prec_ep * rec_ep / max(prec_ep + rec_ep, 1e-9)
    # atraso até a 1ª janela residual do episódio (h)
    j = jt.assign(res=residual)
    atrasos = []
    for ep in episodios.itertuples():
        if ep.episodio in detectados:
            t1 = j.loc[(j["episodio"] == ep.episodio) & j["res"], "janela"].min()
            atrasos.append((t1 - ep.inicio) / pd.Timedelta("1h"))
    met = dict(precisao_jan=p, recall_jan=r, f1_jan=f, pr_auc_jan=average_precision_score(y, razao.to_numpy()[ok]),
               alarmes_limpas=residual[lab == -1].mean(), recall_ep=rec_ep, precisao_ep=prec_ep, f1_ep=f1_ep,
               fp_sequencias=fp_seq, n_episodios=len(com_janela),
               atraso_mediano_h=float(np.median(atrasos)) if atrasos else np.nan)
    detect = episodios.assign(detectado=episodios["episodio"].isin(detectados),
                              avaliavel=episodios["episodio"].isin(com_janela))
    return met, detect, residual


def consultas(js, residual, zs, zn, episodios, lim_teste, janela, poucos, semente, horas_antes=HORAS_ANTES):
    """Simula incidentes registrados durante cada episódio e consultas de controle em períodos limpos.
    ``js`` é a série completa da semente (ordenada); ``residual``, ``zs`` (z com sinal) e ``zn`` (escore
    normalizado por atributo) alinhados a ela."""
    rng = np.random.default_rng(5000 + semente)
    passo, H = pd.Timedelta(janela).to_timedelta64(), pd.Timedelta(hours=horas_antes)
    seq = sequencias(residual, js["recurso"])
    ini_seq = pd.Series(js["janela"].to_numpy()).groupby(seq).min()
    esc = zn.max(axis=1).to_numpy()
    lab = js["episodio"].to_numpy()
    jan = js["janela"].to_numpy()
    rec = js["recurso"].to_numpy()
    linhas = []

    def analisa(r, t_reg):
        m = np.flatnonzero((rec == r) & (jan >= np.datetime64(t_reg - H)) & (jan + passo <= np.datetime64(t_reg)))
        res = m[residual[m]]
        if len(res) == 0:
            return dict(estado="Normal", inicio_est=pd.NaT, lado_pred=None, regra=None, n_res=0, idx=m)
        s_ult = seq[res[-1]]
        da_seq = res[seq[res] == s_ult]
        top = da_seq[np.argmax(esc[da_seq])]
        lado, regra = lado_provavel(zn.iloc[top], zs.iloc[top], js.iloc[top], r in poucos)
        return dict(estado="Residual", inicio_est=ini_seq[s_ult], lado_pred=lado, regra=regra, n_res=len(res), idx=m)

    for ep in episodios.itertuples():
        t_reg = ep.inicio + pd.Timedelta(hours=int(rng.integers(1, ep.duracao_h + 1)))
        a = analisa(ep.recurso, t_reg)
        erro = (a["inicio_est"] - ep.inicio) / pd.Timedelta("1h") if a["estado"] == "Residual" else np.nan
        linhas.append(dict(tipo="incidente", episodio=ep.episodio, recurso=ep.recurso, tipo_falha=ep.tipo,
                           intensidade=ep.intensidade, lado_real=ep.lado, t_registro=t_reg, estado=a["estado"],
                           erro_inicio_h=erro, lado_pred=a["lado_pred"], regra=a["regra"]))
        # consulta de controle no mesmo recurso, sem nenhuma janela rotulada no período analisado
        l = lim_teste.loc[ep.recurso]
        horas = pd.date_range(l["inicio"] + H, l["fim"], freq="1h")
        for _ in range(200):
            t_c = horas[rng.integers(len(horas))]
            c = analisa(ep.recurso, t_c)
            if len(c["idx"]) and (lab[c["idx"]] == -1).all():
                linhas.append(dict(tipo="controle", episodio=ep.episodio, recurso=ep.recurso, tipo_falha=None,
                                   intensidade=None, lado_real=None, t_registro=t_c, estado=c["estado"],
                                   erro_inicio_h=np.nan, lado_pred=c["lado_pred"], regra=c["regra"]))
                break
    return pd.DataFrame(linhas)

## 9. Estilo dos gráficos
Paleta categórica validada (ordem fixa, segura para daltonismo nos três primeiros pares) e rampa sequencial azul. Texto em tons neutros; grade discreta; linhas finas.

In [ ]:
CORES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
AZUL = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]
TXT1, TXT2, TXT3, GRADE, CINZA = "#0b0b0b", "#52514e", "#8a8983", "#e4e3df", "#b8b7b1"
CMAP_SEQ = LinearSegmentedColormap.from_list("azul", ["#f7f9fc"] + AZUL)
COR_DIA = {"util": CORES[0], "sabado": CORES[1], "domingo_feriado": CORES[2]}

mpl.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 8, "axes.titlesize": 8.5, "axes.titleweight": "bold",
    "axes.labelsize": 8, "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7,
    "axes.edgecolor": CINZA, "axes.labelcolor": TXT2, "xtick.color": TXT2, "ytick.color": TXT2,
    "text.color": TXT1, "axes.titlecolor": TXT1, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRADE, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "lines.linewidth": 1.5, "figure.dpi": 110, "savefig.bbox": "tight", "savefig.dpi": 220,
    "pdf.fonttype": 42, "legend.frameon": False, "figure.facecolor": "white", "axes.facecolor": "white",
})


def salvar(fig, nome):
    fig.savefig(FIGURAS / f"{nome}.pdf")
    fig.savefig(FIGURAS / f"{nome}.png")


def salvar_json(obj, nome):
    (RESULTADOS / nome).write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=str))

print("00_funcoes carregado · raiz:", RAIZ)